# Plate Recognition

This notebook trains a YOLOv8n model to find the licence plate in a photo of a car, then reads the plate number with OCR (EasyOCR). It is meant for the camera at the car park entrance and exit.

## Step 1: Install the libraries and load the dataset

On Colab, compress the `datasets` folder into `datasets.zip` and drag it into the Files panel on the left before running this cell. When running locally, keep this notebook next to the `datasets` folder.

In [ ]:
!pip install -q ultralytics easyocr

import os
import zipfile

# Local: keep this notebook next to the "datasets" folder.
# Colab: right-click the "datasets" folder, choose Compress to ZIP, then drag datasets.zip
#        into the Files panel on the left (or upload it when asked below).
if not os.path.exists("datasets"):
    zip_name = "datasets.zip"
    if not os.path.exists(zip_name):
        from google.colab import files
        zip_name = list(files.upload().keys())[0]
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(".")

path = os.path.abspath(".")

## Step 2: Show the dataset path

In [ ]:
print("Dataset path:")
print(path)

### Show the dataset folders

In [ ]:
import os

print(os.listdir(path))

## Step 3: Find the dataset folder automatically

In [ ]:
DATASET_PATH = None

for root, dirs, files in os.walk(path):
    if "car-plate-data" in dirs:
        DATASET_PATH = os.path.join(root, "car-plate-data")
        break

print("Dataset path:")
print(DATASET_PATH)

## Step 4: Check the classes and number of images

The dataset has one class, LicensePlate. Every image has exactly one labelled plate.

In [ ]:
import json

SPLITS = ["train", "valid", "test"]
class_names = ["LicensePlate"]

coco = {}
for split in SPLITS:
    with open(os.path.join(DATASET_PATH, split, "_annotations.coco.json")) as f:
        coco[split] = json.load(f)

# Category 0 is only a Roboflow parent group with no boxes, category 1 is the plate
print("Categories in the dataset:")
for c in coco["train"]["categories"]:
    print(f"  {c['id']}: {c['name']}")

print("\nClasses:")
print(class_names)

print("\nNumber of images and plates:")
for split in SPLITS:
    print(f"{split}: {len(coco[split]['images'])} images, {len(coco[split]['annotations'])} plates")

print("\nTotal number of classes:", len(class_names))

## Step 5: Display sample images

The red box is the labelled licence plate.

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

data = coco["train"]

plt.figure(figsize=(15, 10))

for index, img_info in enumerate(random.sample(data["images"], 6)):
    ax = plt.subplot(2, 3, index + 1)
    ax.imshow(Image.open(os.path.join(DATASET_PATH, "train", img_info["file_name"])).convert("RGB"))

    for a in data["annotations"]:
        if a["image_id"] == img_info["id"]:
            x, y, w, h = a["bbox"]
            ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, color="red", linewidth=2))

    plt.title(class_names[0])
    plt.axis("off")

plt.tight_layout()
plt.show()

## Step 6: Load the dataset in YOLO format

YOLOv8 reads the images from folders, but every box must be written in its own format: one text file per image, with one line per object.

- Every image is resized to 512 x 512 pixels.
- Images are grouped into batches of 16.
- The data is already split into training, validation and test data, so the split is kept.

The converted dataset is saved in `datasets/car-plate-yolo`.

In [ ]:
import shutil

# The converted dataset is saved in datasets/car-plate-yolo
YOLO_PATH = os.path.join(os.path.dirname(DATASET_PATH), "car-plate-yolo")

if os.path.exists(YOLO_PATH):
    shutil.rmtree(YOLO_PATH)  # start fresh every time

for split in SPLITS:
    data = coco[split]
    img_dir = os.path.join(YOLO_PATH, split, "images")
    lbl_dir = os.path.join(YOLO_PATH, split, "labels")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    for img_info in data["images"]:
        W, H = img_info["width"], img_info["height"]

        # YOLO format: one line per object: class_id x_center y_center width height (values 0-1)
        lines = []
        for a in data["annotations"]:
            if a["image_id"] != img_info["id"]:
                continue
            x, y, w, h = a["bbox"]
            lines.append(f"0 {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}")

        shutil.copy(os.path.join(DATASET_PATH, split, img_info["file_name"]), img_dir)
        with open(os.path.join(lbl_dir, os.path.splitext(img_info["file_name"])[0] + ".txt"), "w") as f:
            f.write("\n".join(lines))

    print(f"{split}: {len(os.listdir(img_dir))} images")

## Step 7: Display one training batch

In [ ]:
BATCH_SIZE = 16
train_img_dir = os.path.join(YOLO_PATH, "train", "images")
train_lbl_dir = os.path.join(YOLO_PATH, "train", "labels")

plt.figure(figsize=(10, 10))

for i, file_name in enumerate(random.sample(os.listdir(train_img_dir), 9)):
    ax = plt.subplot(3, 3, i + 1)
    image = Image.open(os.path.join(train_img_dir, file_name)).convert("RGB")
    W, H = image.size
    ax.imshow(image)

    with open(os.path.join(train_lbl_dir, os.path.splitext(file_name)[0] + ".txt")) as f:
        for line in f:
            if not line.strip():
                continue
            class_id, x, y, w, h = line.split()
            x, y, w, h = float(x) * W, float(y) * H, float(w) * W, float(h) * H
            ax.add_patch(patches.Rectangle((x - w / 2, y - h / 2), w, h, fill=False, color="red", linewidth=2))
    plt.axis("off")

plt.tight_layout()
plt.show()

## Step 8: Create the dataset file

`data.yaml` tells YOLO:

- `train`, `val` and `test`: where the training, validation and test images are
- `nc`: the number of classes
- `names`: the class names

In [ ]:
DATA_YAML = os.path.join(YOLO_PATH, "data.yaml")

with open(DATA_YAML, "w") as f:
    f.write(f"path: {YOLO_PATH}\n")
    f.write("train: train/images\nval: valid/images\ntest: test/images\n")
    f.write(f"nc: {len(class_names)}\n")
    f.write(f"names: {class_names}\n")

print(open(DATA_YAML).read())
print("Dataset pipeline is ready.")

## Step 9: Build the AI model (YOLOv8n)

This is the AI model that will find the licence plate in our images. Instead of creating a CNN from scratch, we use transfer learning: YOLOv8n was already trained on the COCO dataset, so it starts with useful knowledge of cars and objects.

The model has three parts. The backbone, pre-trained on COCO, takes the 512x512 input image and extracts features such as edges and character shapes. The neck combines small and large features so that both small and large plates are found. The detection head then predicts a box and a confidence score for every licence plate.

Why YOLOv8n:

- Very fast
- Small model size
- Designed for edge devices
- High accuracy
- Compatible with TensorFlow Lite
- Can be compiled for the Coral Edge TPU

The detector only finds where the plate is. Reading the characters on the plate is done afterwards by OCR in Step 14.

In [ ]:
from ultralytics import YOLO

# Load the pre-trained YOLOv8n model
model = YOLO("yolov8n.pt")

# Show model architecture
model.info()

## Step 10: Set the training settings

YOLO has no separate `compile()` step like TensorFlow. The optimizer, number of epochs and image size are chosen here and passed to training in the next step. The loss function and the metrics (precision, recall and mAP) are built in.

In [ ]:
import torch

# YOLO has no compile() step like TensorFlow, the settings are chosen here instead.
# The loss function (box loss + class loss) and the metrics (precision, recall, mAP) are built in.
TRAIN_SETTINGS = dict(
    data=DATA_YAML,
    epochs=80,          # small dataset (41 training images), so more epochs
    patience=25,        # stop early if validation stops improving for 25 epochs
    imgsz=512,
    batch=BATCH_SIZE,
    optimizer="auto",   # YOLO picks the best optimizer
    device=0 if torch.cuda.is_available() else "cpu",   # use a GPU: on Colab choose Runtime, Change runtime type, T4 GPU
    project="runs",
    name="plate",
    exist_ok=True,
)

print("Model settings are ready.")

## Step 11: Train the AI model

In this step, the AI learns from the training images.

- Training dataset: used to teach the model.
- Validation dataset: used to evaluate the model after each epoch.
- Epochs = 80: the model sees the entire dataset up to 80 times.

The dataset is small (41 training images), so the model trains for more epochs than usual.

During training, YOLO displays something like:

```text
Epoch 1/80
box_loss: 1.52
cls_loss: 2.10
Precision: 0.65
Recall: 0.58
mAP50: 0.61
```

Each epoch shows:

- box_loss: box position error (lower is better)
- cls_loss: class error (lower is better)
- Precision: how many detected boxes are correct
- Recall: how many real plates were found
- mAP50: detection accuracy (higher is better)

At each epoch, mAP50 should increase and the losses should decrease. If the final mAP50 is above 0.90, the model is performing well.

Use a GPU for training (on Colab: Runtime, Change runtime type, T4 GPU). On a CPU it is much slower.

In [ ]:
results = model.train(**TRAIN_SETTINGS)

RUN_DIR = str(results.save_dir)
print("Training results saved in:", RUN_DIR)

## Step 12: Plot training accuracy and loss

mAP50 is the detector's version of accuracy, so it is plotted together with precision and recall.

In [ ]:
import pandas as pd

history = pd.read_csv(os.path.join(RUN_DIR, "results.csv"))
history.columns = history.columns.str.strip()

# Accuracy values (mAP50 is the "accuracy" of a detector)
val_map50 = history["metrics/mAP50(B)"]
val_precision = history["metrics/precision(B)"]
val_recall = history["metrics/recall(B)"]

# Loss values
train_loss = history["train/box_loss"] + history["train/cls_loss"]
val_loss = history["val/box_loss"] + history["val/cls_loss"]

epochs_range = history["epoch"]

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, val_map50, label="Validation mAP50")
plt.plot(epochs_range, val_precision, label="Validation Precision")
plt.plot(epochs_range, val_recall, label="Validation Recall")
plt.title("Validation Accuracy (mAP50)")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, train_loss, label="Training Loss")
plt.plot(epochs_range, val_loss, label="Validation Loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.legend()

plt.show()

## Step 13: Evaluate the model

This is the final performance of the AI model, measured on test images it has never seen before.

There are three important metrics:

- mAP50: how many plates were detected correctly. For example, mAP50 = 0.95 means the model correctly found about 95 out of every 100 plates. Higher is better.
- Precision: how many of the detected boxes are correct. Higher is better.
- Recall: how many of the real plates were found. Higher is better.

If the test mAP50 is high, the model can generalise well to new images.

The test set only has 13 images, so one wrong image changes the score a lot.

In [ ]:
# Evaluate the model on the test dataset
best_model = YOLO(os.path.join(RUN_DIR, "weights", "best.pt"))
metrics = best_model.val(data=DATA_YAML, split="test", imgsz=512)

print("\n========== Model Evaluation ==========")
print(f"Precision : {metrics.box.mp:.3f}")
print(f"Recall    : {metrics.box.mr:.3f}")
print(f"mAP50     : {metrics.box.map50:.3f}")

## Step 14: Test the AI model on one image

This is the inference stage. The model is no longer learning; it is making a prediction on a new image.

The input image is resized to 512x512 and passed through the YOLOv8n model, which returns a bounding box and a confidence score for every licence plate it finds.

By default a random test image is used. To test your own car photo on Colab, set `UPLOAD_OWN_IMAGE = True`.

If a plate is missed, lower `CONF_THRESHOLD`. If the same plate is found twice, `NMS_IOU` removes the duplicate box.

In [ ]:
# False: use a random image from the test set
# True:  upload your own car photo on Colab
UPLOAD_OWN_IMAGE = False

test_img_dir = os.path.join(YOLO_PATH, "test", "images")

if UPLOAD_OWN_IMAGE:
    from google.colab import files
    uploaded = files.upload()
    img_path = list(uploaded.keys())[0]
else:
    img_path = os.path.join(test_img_dir, random.choice(os.listdir(test_img_dir)))

print("Image:", img_path)

In [ ]:
# Only boxes with at least this confidence are kept (0.25 is the YOLO default).
# If a plate is missed, try a lower value such as 0.15.
CONF_THRESHOLD = 0.25

# Boxes that overlap more than 30% are treated as the same plate, only the most confident one is kept
NMS_IOU = 0.3

result = best_model.predict(img_path, conf=CONF_THRESHOLD, iou=NMS_IOU, imgsz=512, verbose=False)[0]

if len(result.boxes) == 0:
    # Show how close the model came, to help choose CONF_THRESHOLD
    low = best_model.predict(img_path, conf=0.01, imgsz=512, verbose=False)[0]
    best_conf = max(low.boxes.conf.tolist()) * 100 if len(low.boxes) else 0
    print(f"No plate above {CONF_THRESHOLD * 100:.0f}% confidence. The best guess was {best_conf:.2f}%.")
    print("If that guess is the plate, lower CONF_THRESHOLD. If it is very low, the model needs more training images like this one.")

plt.figure(figsize=(7, 7))
ax = plt.gca()
ax.imshow(Image.open(img_path).convert("RGB"))

for box in result.boxes:
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="red", linewidth=2))
    ax.text(x1, y1 - 6, f"LicensePlate {float(box.conf) * 100:.2f}%", color="red", fontsize=10, weight="bold")

plt.title(f"Plates found: {len(result.boxes)}")
plt.axis("off")
plt.show()

### Read the plate number with OCR

The plate is cut out of the image and given to EasyOCR, a ready-made text reader that does not need training.

Malaysian plates are letters, then numbers, then sometimes one more letter (for example VBM 6529 or WC 679 T). OCR sometimes confuses look-alike characters such as O and 0, or S and 5, so the result is corrected using this format.

In [ ]:
import re
import numpy as np
import easyocr

reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available(), verbose=False)

PLATE_FORMAT = re.compile(r"^[A-Z]{1,3}[0-9]{1,4}[A-Z]?$")
TO_LETTER = {"0": "O", "1": "I", "2": "Z", "5": "S", "6": "G", "8": "B"}
TO_DIGIT = {"O": "0", "D": "0", "Q": "0", "I": "1", "L": "1", "Z": "2", "S": "5", "G": "6", "B": "8", "T": "7"}


def read_plate(crop):
    pieces = reader.readtext(crop, allowlist="ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789")
    if not pieces:
        return "", 0.0

    # Put the text pieces in reading order: top line first, then left to right
    pieces.sort(key=lambda p: (p[0][0][1] + p[0][2][1]) / 2)
    lines = []
    for p in pieces:
        center_y = (p[0][0][1] + p[0][2][1]) / 2
        height = p[0][2][1] - p[0][0][1]
        if lines and abs(center_y - lines[-1][0]) < height / 2:
            lines[-1][1].append(p)
        else:
            lines.append([center_y, [p]])
    ordered = [p for _, line in lines for p in sorted(line, key=lambda p: p[0][0][0])]

    text = "".join(p[1] for p in ordered)
    confidence = float(np.mean([p[2] for p in ordered]))
    return text, confidence


def clean_plate(text):
    text = re.sub(r"[^A-Z0-9]", "", text.upper())

    # Try every way to split the text into letters + numbers + optional letter,
    # fix look-alike characters, and keep the split that needs the fewest changes
    best, best_changes = text, None
    for n_prefix in (3, 2, 1):
        for n_suffix in (0, 1):
            prefix = text[:n_prefix]
            digits = text[n_prefix:len(text) - n_suffix]
            suffix = text[len(text) - n_suffix:]

            new_prefix = "".join(TO_LETTER.get(c, c) for c in prefix)
            new_digits = "".join(TO_DIGIT.get(c, c) for c in digits)
            new_suffix = "".join(TO_LETTER.get(c, c) for c in suffix)
            changes = sum(a != b for a, b in zip(prefix + digits + suffix, new_prefix + new_digits + new_suffix))

            # Malaysian plate numbers never start with 0 (an extra 0 is usually the edge of the plate)
            changes += len(new_digits) - len(new_digits.lstrip("0"))
            new_digits = new_digits.lstrip("0")

            candidate = new_prefix + new_digits + new_suffix
            if PLATE_FORMAT.match(candidate) and (best_changes is None or changes < best_changes):
                best, best_changes = candidate, changes
    return best


def format_plate(plate):
    # VBM6529 -> VBM 6529, WC679T -> WC 679 T
    match = re.match(r"^([A-Z]+)([0-9]+)([A-Z]?)$", plate)
    return " ".join(part for part in match.groups() if part) if match else plate


image = np.array(Image.open(img_path).convert("RGB"))

for box in result.boxes:
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
    crop = image[max(0, y1 - 3):y2 + 3, max(0, x1 - 3):x2 + 3]

    raw_text, ocr_conf = read_plate(crop)
    plate = clean_plate(raw_text)

    plt.figure(figsize=(5, 2))
    plt.imshow(crop)
    plt.title(format_plate(plate) if plate else "(not readable)")
    plt.axis("off")
    plt.show()

    print("OCR raw text   :", raw_text)
    print("Plate number   :", format_plate(plate))
    print("Valid format   :", "Yes" if PLATE_FORMAT.match(plate) else "No")
    print(f"OCR confidence : {ocr_conf * 100:.2f}%")

### Read the plates in all test images

This is what the entrance camera does for every car: find the plate, read it and record the time.

In [ ]:
from datetime import datetime


def recognise_plate(image_path):
    result = best_model.predict(image_path, conf=CONF_THRESHOLD, iou=NMS_IOU, imgsz=512, verbose=False)[0]
    if len(result.boxes) == 0:
        return None, "", "", 0.0

    box = result.boxes[int(result.boxes.conf.argmax())]  # the most confident plate
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
    image = np.array(Image.open(image_path).convert("RGB"))
    crop = image[max(0, y1 - 3):y2 + 3, max(0, x1 - 3):x2 + 3]

    raw_text, ocr_conf = read_plate(crop)
    return crop, raw_text, clean_plate(raw_text), ocr_conf


rows = []
test_files = sorted(os.listdir(test_img_dir))

plt.figure(figsize=(16, 10))
for i, file_name in enumerate(test_files):
    crop, raw_text, plate, ocr_conf = recognise_plate(os.path.join(test_img_dir, file_name))
    rows.append({
        "Image": file_name[:20],
        "Plate Number": format_plate(plate) if plate else "-",
        "Valid Format": "Yes" if PLATE_FORMAT.match(plate) else "No",
        "OCR Confidence": f"{ocr_conf * 100:.1f}%",
        "Time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    })

    if crop is not None and i < 12:
        plt.subplot(4, 3, i + 1)
        plt.imshow(crop)
        plt.title(format_plate(plate) if plate else "(not readable)")
        plt.axis("off")

plt.tight_layout()
plt.show()

valid = sum(1 for r in rows if r["Valid Format"] == "Yes")
print(f"Plates read in a valid Malaysian format: {valid} / {len(rows)}")

plate_log = pd.DataFrame(rows)
plate_log

## Step 15: Save the trained model

At this point training is finished. The model is saved so it can be used later without training again.

The `.pt` file contains:

- the model architecture
- the learned weights
- the class names

EasyOCR does not need to be saved; it downloads its own model the first time it is used.

In [ ]:
# Save the trained model
shutil.copy(os.path.join(RUN_DIR, "weights", "best.pt"), "plate_yolov8n.pt")

print("Model saved successfully!")

## Step 16: Check that the model was saved

List the files in the current folder to make sure the model file was created.

In [ ]:
import os

print(os.listdir())

try:
    from google.colab import files
    files.download("plate_yolov8n.pt")
except ImportError:
    pass  # running locally, the file is already in this folder

## Step 17: Measure the inference time

Run the saved model on the same image and measure how long it takes to find the plate (detection) and to read it (OCR).

In [ ]:
import time

saved_model = YOLO("plate_yolov8n.pt")
print(f"Processing image: {img_path}")

saved_model.predict(img_path, imgsz=512, verbose=False)  # warm-up run (the first run is always slower)

# Time 1: finding the plate
start_time = time.perf_counter()
result = saved_model.predict(img_path, conf=CONF_THRESHOLD, iou=NMS_IOU, imgsz=512, verbose=False)[0]
detect_time = (time.perf_counter() - start_time) * 1000

# Time 2: reading the plate
if len(result.boxes) > 0:
    x1, y1, x2, y2 = map(int, result.boxes[0].xyxy[0].tolist())
    crop = image[max(0, y1 - 3):y2 + 3, max(0, x1 - 3):x2 + 3]
    start_time = time.perf_counter()
    raw_text, _ = read_plate(crop)
    ocr_time = (time.perf_counter() - start_time) * 1000
else:
    raw_text, ocr_time = "", 0.0

print("========== COLAB INFERENCE ==========")
print(f"Plate number     : {format_plate(clean_plate(raw_text))}")
print(f"Detection Time   : {detect_time:.2f} ms")
print(f"OCR Time         : {ocr_time:.2f} ms")
print(f"Total Time       : {detect_time + ocr_time:.2f} ms")

### More accurate timing measurement

A single measurement is not reliable. The same image is run 20 times and the average time is calculated.

In [ ]:
detect_times = []
ocr_times = []

for _ in range(20):
    start_time = time.perf_counter()
    result = saved_model.predict(img_path, conf=CONF_THRESHOLD, iou=NMS_IOU, imgsz=512, verbose=False)[0]
    detect_times.append((time.perf_counter() - start_time) * 1000)

    if len(result.boxes) > 0:
        x1, y1, x2, y2 = map(int, result.boxes[0].xyxy[0].tolist())
        crop = image[max(0, y1 - 3):y2 + 3, max(0, x1 - 3):x2 + 3]
        start_time = time.perf_counter()
        read_plate(crop)
        ocr_times.append((time.perf_counter() - start_time) * 1000)

average_total = np.mean(detect_times) + (np.mean(ocr_times) if ocr_times else 0)

print("========== COLAB INFERENCE RESULTS ==========")
print(f"Average Detection Time : {np.mean(detect_times):.2f} ms")
print(f"Average OCR Time       : {np.mean(ocr_times) if ocr_times else 0:.2f} ms")
print(f"Average Total Time     : {average_total:.2f} ms")
print(f"Cars Per Second        : {1000 / average_total:.1f}")